# End-to-End Tabular Contest Practice

**Topic:** Past Problem Lab &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `train_df` (3,500 rows): student-competition entries with numeric features (some `NaN`s), a categorical `division` column, and the 0/1 label `advanced` (did the entry advance to the next round).
- `test_df` (1,200 rows): same columns without the label. `y_test` is used by the evaluation cell.
- Signal is spread across several features and one interaction. No single column gets you there.

## Your task

Run the full workflow:

1. Explore and clean (impute `NaN`s, encode `division`).
2. Establish a baseline, then improve it with a stronger model or feature work.
3. Set `predictions` to your 0/1 predictions for `test_df`.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np
import pandas as pd

def make(n, seed, with_label):
    r = np.random.default_rng(seed)
    hours = r.gamma(3.0, 12.0, n)
    score1 = r.uniform(0, 100, n)
    score2 = r.uniform(0, 100, n)
    experience = r.integers(0, 6, n).astype(float)
    division = r.choice(["junior", "senior", "open"], n, p=[0.4, 0.4, 0.2])
    logit = (
        0.044 * hours + 0.048 * score1 + 0.034 * score2
        + 0.60 * experience + 0.024 * (score1 > 70) * score2
        + np.select([division == "junior"], [-1.0], default=0.4) - 8.8
    )
    y = (1 / (1 + np.exp(-logit)) > r.uniform(0, 1, n)).astype(int)
    df = pd.DataFrame({
        "practice_hours": hours.round(1), "theory_score": score1.round(1),
        "notebook_score": score2.round(1), "years_experience": experience,
        "division": division,
    })
    for col in ["practice_hours", "theory_score", "years_experience"]:
        df.loc[r.random(n) < 0.08, col] = np.nan
    if with_label:
        df["advanced"] = y
    return df, y

train_df, _ = make(3500, 171, True)
test_df, y_test = make(1200, 172, False)
print(train_df.dtypes)
train_df.head()


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.78

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

The full contest loop in one pipeline: median-impute the numeric columns, one-hot the division, and hand the result to gradient boosting (which finds the theory_score x notebook_score interaction on its own). Validate before predicting.

```python
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric = ["practice_hours", "theory_score", "notebook_score", "years_experience"]
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["division"]),
])
model = Pipeline([("prep", prep), ("clf", HistGradientBoostingClassifier(random_state=0))])

X = train_df.drop(columns=["advanced"])
y = train_df["advanced"]
print("majority-class baseline accuracy:", max(y.mean(), 1 - y.mean()).round(3))
print("CV accuracy:", cross_val_score(model, X, y, cv=5).mean().round(3))

model.fit(X, y)
predictions = model.predict(test_df)
```
